# AGC 2027 — Optimasi Titik Distribusi Air Bersih Katongan (Skenario)

**Hasil awal dari paket:** 11 pusat pencarian model (95,84% dalam radius 500 m garis lurus), ditambah 3 pusat temporer fase II (98,16%).

**Peringatan:** Data bukan titik drop air eksisting; bukan lokasi pemasangan final; bukan *road-network routing*. Lokasi harus diaudit akses truk, lahan, sumber air, dan kebutuhan penerima. **Bangunan ≠ rumah tangga.**

Notebook ini menjalankan kembali source code transparan di dalam ZIP paket. Upload ZIP utuh yang telah diberikan di chat.

In [ ]:
# 1) Install dependensi. Colab membutuhkan internet saat pertama menjalankan.
%pip -q install "numpy>=1.26" "pandas>=2.2" "geopandas>=0.14" "shapely>=2" "pyproj>=3.6" "scipy>=1.11" "matplotlib>=3.8"

In [ ]:
# 2) Upload paket ZIP AGC2027_Katongan_Proposed_Water_Points_v1.zip
from google.colab import files
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
assert len(zip_names) >= 1, 'Upload paket ZIP dulu.'
import zipfile, pathlib
workdir = pathlib.Path('/content/agc2027_prospective')
workdir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_names[0], 'r') as zf:
    zf.extractall(workdir)
packages = list(workdir.rglob('src/optimize_katongan.py'))
assert packages, 'Source tidak ditemukan; pastikan upload ZIP paket lengkap.'
root = packages[0].parents[1]
print('Project root:', root)
print('Input files:', [p.name for p in (root/'inputs').glob('*')])

In [ ]:
# 3) Jalankan ulang 342 kandidat, p=4..16, optimasi dan semua ekspor GeoJSON/CSV
import subprocess, sys
result = subprocess.run([sys.executable, str(root/'src/optimize_katongan.py')], text=True, capture_output=True)
print(result.stdout[-7000:])
if result.returncode:
    print(result.stderr[-5000:])
assert result.returncode == 0, 'Model gagal dieksekusi.' 

In [ ]:
# 4) QA otomatis
check = subprocess.run([sys.executable, str(root/'tests/test_outputs.py')], capture_output=True, text=True)
print(check.stdout, check.stderr)
assert check.returncode == 0

import json, pandas as pd
summary = json.loads((root/'outputs/model_summary.json').read_text())
print('Lokasi utama:', summary['selected_p'])
print('Cakupan utama 500 m:', round(summary['coverage_500m_pct'],2), '%')
print('Tambahan temporer:', summary['phase2_supplementary_count'])
print('Cakupan fase II:', round(summary['phase2_coverage_500m_pct'],2), '%')
display(pd.read_csv(root/'outputs/proposed_site_search_centres_11.csv')[['site_id','latitude','longitude','nearest_assigned_buildings','proxy_demand_L_day_50pct_15Lpcd']])

In [ ]:
# 5) Peta dan grafik skenario
from IPython.display import Image, display
img = root/'outputs/proposed_site_search_map.png'
display(Image(filename=str(img)))
import matplotlib.pyplot as plt
comp = pd.read_csv(root/'outputs/scenario_site_count_comparison.csv')
fig, ax = plt.subplots(figsize=(9,4.5))
for mode, df in comp.groupby('method'):
    ax.plot(df.site_count, df.coverage_500m_pct,marker='o',label=mode)
ax.axhline(95,linestyle='--',linewidth=1.2,label='Target skenario 95%')
ax.set(xlabel='Jumlah pusat layanan kandidat',ylabel='Cakupan bangunan dalam 500 m (%)',title='Trade-off jumlah fasilitas vs cakupan garis lurus')
ax.legend(); ax.grid(alpha=0.2);plt.show()

In [ ]:
# 6) Unduh keluaran paling penting sebagai ZIP untuk SI
import os, zipfile
outzip = pathlib.Path('/content/AGC2027_Katongan_Waterpoints_Outputs_for_SI.zip')
with zipfile.ZipFile(outzip,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in (root/'outputs').rglob('*'):
        if f.is_file(): z.write(f,arcname=f.name)
    for f in [root/'README_START_HERE.md',root/'docs/SI_IMPLEMENTATION_CONTRACT.md',root/'docs/METHODOLOGY_AND_LIMITATIONS.md']:
        z.write(f,arcname=f.name)
print(outzip, outzip.stat().st_size)
files.download(str(outzip))